<a href="https://colab.research.google.com/github/azam-hussain-ml/Starter-Notebook-flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/azam-hussain-ml/Starter-Notebook-flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

Clones the repo on Colab so the starter CSV is there. Running locally it just walks up to the repo root.

In [1]:
import os, sys, subprocess

repo_url = "https://github.com/azam-hussain-ml/Starter-Notebook-flyrank-ml-internship"
repo_dir = "Starter-Notebook-flyrank-ml-internship"
csv_path = "data/raw/content_refresh_anonymized.csv"

if "google.colab" in sys.modules:
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_dir], check=True)
    os.chdir(repo_dir)
else:
    # notebook lives in work/notebooks/ so climb out to the root
    for _ in range(4):
        if os.path.exists(csv_path):
            break
        os.chdir("..")

print("working dir:", os.getcwd())
assert os.path.exists(csv_path), "can't find the starter CSV"

working dir: /content/Starter-Notebook-flyrank-ml-internship


In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

df = pd.read_csv(csv_path)
print(df.shape)

(30000, 44)


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane 2 — Refresh / Content Opportunity Scoring. This is a ranking task.**

The decision isn't "is this page declining." It's "which 50 pages does a strategist open first on Monday." Capacity is the fixed thing here: 30,000 pages in the inventory, about 50 reviews in a human week.

That's what rules the other options out. Classification answers yes/no, and here it says yes 16,262 times for 50 slots. Correct, useless. Clustering would tell me what kinds of pages exist, which I might want later for reason codes, but it doesn't give me a first, second, third. Ranking gives the strategist exactly what they consume: an ordered list where only the top ever gets read.

One thing I want to be careful about, because I had it confused at first. I'll probably train a classifier and use its predicted probability as the score. That's normal. What makes this a ranking problem isn't the algorithm, it's the evaluation. I'm judged on 50 rows, not on 30,000 labels.

Worth spelling out what happens after the queue, since the ordering isn't the end of it. The strategist opens a page and picks one of five actions: refresh it, expand it, protect it, prune it, or leave it on monitoring. My score doesn't choose the action — it only decides who gets looked at, and every row carries a reason code so the strategist can see why it surfaced and overrule it. Nothing I build touches the site.

The cell below is the reason. I stacked three filters I could defend to a client and the queue is still 9,329 pages, which is 187 weeks of review capacity. There's no filter that gets this down to 50. Something has to order the survivors, and that ordering is the deliverable.

In [3]:
review_budget = 50   # pages a strategist can get through in a week

declining = df["trend_direction"] == "down"
has_traffic = df["impressions_90d"] >= 100
visible = df["avg_position"].between(1, 20)   # position 0 means no data, don't want those

print("all pages:", len(df))
print("declining:", declining.sum())
print("declining + 100 impressions:", (declining & has_traffic).sum())
print("declining + 100 impressions + position 1-20:", (declining & has_traffic & visible).sum())

queue = (declining & has_traffic & visible).sum()
print()
print(f"so the shortlist is {queue:,} pages against a budget of {review_budget}")
print(f"that's {queue / review_budget:.0f} weeks of work")

all pages: 30000
declining: 16262
declining + 100 impressions: 13152
declining + 100 impressions + position 1-20: 9329

so the shortlist is 9,329 pages against a budget of 50
that's 187 weeks of work


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Two answers here, because the honest one changes in Week 3.

The label that ships with the starter file is a proxy, and it's defined by a rule. `is_declining_label = (trend_direction == "down")`, and `trend_direction` is just a threshold on `trend_pct`: down means impressions fell more than 20% against the previous 30 days. I didn't want to take that on trust, so I tried to rebuild it from the threshold alone. It reproduces on 29,996 of 30,000 rows. The 4 that miss all sit exactly on -20.0, so their cutoff is `<= -20` and mine was `< -20`. That's the whole difference. A model trained on this label is partly learning somebody's -20% cutoff rather than learning which pages are at risk.

What I actually want to predict is a forward-window outcome: for a page with real exposure, did impressions fall more than 20% over the next 30 days. Column name `declined_next_30d`, with a floor of 100 impressions in the feature window so I'm not labelling noise on a page eleven people visited.

I can sketch that column now by reading the file's two windows as before and after: `*_prev_30d` is the feature window, `*_last_30d` is the target window.

Being straight about this: arithmetically that's the same subtraction that produced `trend_pct`. The number doesn't change. What changes is what I'm allowed to use as a feature. Once the last 30 days is the target window, every 90-day column overlaps it, so they're all out. That's 20 columns gone on leakage grounds, leaving me 14 feature candidates out of 44 — the rest are ids, provider/model metadata, and tier columns that just repeat numbers I already have. Which is exactly why this lane needs the warehouse daily table from Week 3. There I can separate the two windows properly instead of borrowing them from a snapshot.

Flag I'm carrying forward rather than hiding: `content_age_days` and `days_since_last_update` are measured at export time, which is the end of my target window, not the start. They're near-static so I'm keeping them for now, but they're mildly contaminated and I'll re-derive both as of the decision date once I'm on the daily table.

In [4]:
# can I rebuild the shipped label from one threshold? if yes, it's a rule not an outcome
rebuilt = (df["trend_pct"] < -20) & df["trend_pct"].notna()

print("matches:", (rebuilt == declining).sum(), "of", len(df))
print("misses:", (rebuilt != declining).sum())
print()
print("what are the misses?")
print(df.loc[rebuilt != declining, ["trend_pct", "trend_direction"]])

matches: 29996 of 30000
misses: 4

what are the misses?
       trend_pct trend_direction
3792       -20.0            down
6972       -20.0            down
19335      -20.0            down
23496      -20.0            down


In [5]:
# the target I actually want, sketched with the two windows already in the file
min_impressions = 100
drop_threshold = -20

eligible = df["impressions_prev_30d"] >= min_impressions
pct_change = (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"] * 100
declined_next_30d = (pct_change < drop_threshold).astype(int)

print("eligible pages:", eligible.sum(), "of", len(df))
print("base rate:", round(declined_next_30d[eligible].mean(), 3))

eligible pages: 18010 of 30000
base rate: 0.616


In [13]:
# what does the framing cost me? every column whose window touches the target window is out
out = ["trend_pct", "trend_direction",
       "impressions_90d", "clicks_90d", "sessions_90d", "pageviews_90d", "users_90d",
       "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
       "days_with_impressions", "days_with_sessions",
       "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
       "impressions_last_30d", "clicks_last_30d", "sessions_last_30d"]

keep = ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
        "content_age_days", "days_since_last_update",
        "word_count", "char_count", "search_volume", "competition", "cpc",
        "content_type", "main_intent", "competition_level", "age_tier_order"]

print("dropped:", len(out), "columns (leakage - they span the target window)")
print("left to work with:", len(keep), "of", df.shape[1])

dropped: 20 columns (leakage - they span the target window)
left to work with: 14 of 44


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@50. Of the top 50 pages in my queue, how many actually declined. K is 50 because that's
the strategist's week, not a number I get to tune.

Not accuracy. The base rate below is about 62%, so answering "declining" for everything scores
0.62 while doing no work at all. Any metric a constant can win isn't measuring anything.

Not AUC as the headline either. AUC scores the whole 18,000-row ordering and nobody reads past row
50. I'll keep it around as a sanity check.

What good means, written down now so I can't move the goalposts once I've seen a score:

1. Beat the base rate. That's the do-nothing floor.
2. Beat the best single-signal rule I can find, otherwise the model hasn't earned its complexity.
3. Beat both across clients, not on average. A queue that works for four clients and falls over on
   the fifth isn't shippable.

The caveat that comes with a top-50 metric: 50 items means a standard error of about ±0.07, so one
split can move 7 points on noise alone. I report the mean over 5 client-grouped folds plus the
spread, and I don't get excited about a gap smaller than the spread.

In [7]:
def precision_at_k(y, scores, k=50):
    # sort by score, take the top k, what fraction of them are positive
    y = np.asarray(y)
    top = np.argsort(-np.asarray(scores, dtype=float))[:k]
    return y[top].mean()


base_rate = declined_next_30d[eligible].mean()
se = np.sqrt(base_rate * (1 - base_rate) / review_budget)

print("do-nothing floor:", round(base_rate, 3))
print("standard error at k=50:", round(se, 3))
print("-> anything under about", round(2 * se, 2), "on a single split is noise")

do-nothing floor: 0.616
standard error at k=50: 0.069
-> anything under about 0.14 on a single split is noise


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one page, for one client, in one 90-day snapshot.

Not a page-day, not a keyword, not a client. The grain has to match the decision, and the decision
happens one page at a time. A strategist opens a page.

Below is my lane's slice: ids kept for grouping only, the feature-window columns, the eligibility
flag, and the target column sitting where it'll sit. The grain probe is the assert. If one row
really is one page then `content_id` has to be unique, and it is, 30,000 of 30,000.

Worth writing down now: this grain changes in Week 3. On `fact_content_daily_performance` the
natural grain is page × decision date, so one page contributes many rows and the split has to hold
out clients *and* time. Otherwise the same page turns up on both sides of the split under two
different dates.

In [8]:
ids = ["content_id", "client_id"]        # grouping only, never features
features = ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
            "content_age_days", "days_since_last_update", "word_count",
            "search_volume", "competition", "content_type", "main_intent"]

lane_df = df[ids + features].copy()
lane_df["is_eligible"] = eligible.astype(int)
lane_df["declined_next_30d"] = declined_next_30d

# grain check - one row should be one page
print("rows:", len(lane_df))
print("unique content_id:", lane_df["content_id"].nunique())
print("clients:", lane_df["client_id"].nunique())
assert lane_df["content_id"].duplicated().sum() == 0, "grain is broken, content_id repeats"

rows: 30000
unique content_id: 30000
clients: 32


In [9]:
lane_df[lane_df["is_eligible"] == 1].head(8)

,content_id,client_id,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,days_since_last_update,word_count,search_volume,competition,content_type,main_intent,is_eligible,declined_next_30d
0,content_304f48230142,client_f369cb89fc,987,13,9,187,20,3221.0,10.0,0.67,keyword article,transactional,1,1
1,content_a1fb4e703a9e,client_4e07408562,5915,1,2,445,25,2481.0,90.0,0.01,keyword article,informational,1,1
2,content_9aa793d4d895,client_7f2253d7e2,6089,3,3,141,20,3515.0,0.0,0.00,keyword article,informational,1,1
3,content_331d6c4de07b,client_19581e27de,4206,17,26,463,22,NaN,10.0,0.00,keyword article,commercial,1,0
4,content_d99b7a2d90ca,client_3fdba35f04,6452,2,9,263,14,2803.0,0.0,0.00,keyword article,informational,1,1
5,content_d4084a4bc775,client_f369cb89fc,1009,1,1,147,20,3080.0,720.0,1.00,keyword article,transactional,1,1
7,content_a63219c6e95a,client_19581e27de,632,0,4,445,22,NaN,590.0,0.44,keyword article,commercial,1,0
8,content_5e6c160719bc,client_6208ef0f77,13828,8,14,90,20,3807.0,0.0,0.00,keyword article,informational,1,1


In [10]:
# how the target splits on the rows I'd actually be scoring
lane_df.loc[lane_df["is_eligible"] == 1, "declined_next_30d"].value_counts().rename({0: "held or grew", 1: "declined"})

,count
declined_next_30d,
declined,11090
held or grew,6920


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

I didn't want to just assert this, so I tested it. Five single-signal rules, each one a sorted
column and the kind of thing you'd write as an if-statement, against a small gradient-boosted
scorer on the same feature-window columns. Five folds grouped by client, so no client appears in
both training and test.

In [11]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold

d = df[eligible].reset_index(drop=True)
y = declined_next_30d[eligible].reset_index(drop=True)

num_cols = ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
            "content_age_days", "days_since_last_update", "word_count", "char_count",
            "search_volume", "competition", "cpc", "age_tier_order"]
cat_cols = ["content_type", "main_intent", "competition_level"]

X = d[num_cols].copy()
for c in cat_cols:
    X[c] = d[c].astype("category")

# the if-statements, written as sortable columns. minus sign = smaller is worse
rules = {
    "biggest prev-30d exposure": d["impressions_prev_30d"],
    "stalest page": d["days_since_last_update"],
    "oldest page": d["content_age_days"],
    "lowest prev-30d CTR": -(d["clicks_prev_30d"] / d["impressions_prev_30d"]),
    "thinnest page": -d["word_count"].fillna(d["word_count"].median()),
}

results = {name: [] for name in list(rules) + ["-- the model --", "do-nothing (base rate)"]}

for train_idx, test_idx in GroupKFold(n_splits=5).split(X, y, groups=d["client_id"]):
    model = HistGradientBoostingClassifier(
        categorical_features=[X.columns.get_loc(c) for c in cat_cols],
        max_iter=200, learning_rate=0.06, random_state=0)
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    proba = model.predict_proba(X.iloc[test_idx])[:, 1]

    y_test = y.iloc[test_idx].values
    results["-- the model --"].append(precision_at_k(y_test, proba))
    results["do-nothing (base rate)"].append(y_test.mean())
    for name, signal in rules.items():
        results[name].append(precision_at_k(y_test, signal.values[test_idx]))

scores = pd.DataFrame({
    "mean P@50": {k: np.mean(v) for k, v in results.items()},
    "worst fold": {k: min(v) for k, v in results.items()},
    "best fold": {k: max(v) for k, v in results.items()},
}).sort_values("mean P@50").round(3)

scores

,mean P@50,worst fold,best fold
biggest prev-30d exposure,0.492,0.40,0.640
oldest page,0.516,0.24,0.800
thinnest page,0.596,0.38,0.860
do-nothing (base rate),0.634,0.51,0.768
stalest page,0.680,0.46,0.860
lowest prev-30d CTR,0.688,0.48,0.840
-- the model --,0.744,0.66,0.800


In [12]:
# do the rules even agree on which pages to review?
import itertools

top50 = {name: set(d.loc[np.argsort(-signal.values)[:50], "content_id"]) for name, signal in rules.items()}
overlaps = [len(top50[a] & top50[b]) for a, b in itertools.combinations(top50, 2)]

print("overlap between any two rules' top 50:")
print("  min", min(overlaps), "| max", max(overlaps), "| mean", round(np.mean(overlaps), 1), "pages out of 50")

overlap between any two rules' top 50:
  min 0 | max 5 | mean 0.6 pages out of 50


### Reading it back

The mean gap is real but smaller than I expected. Model 0.744, do-nothing floor 0.634, best single
rule 0.688. Eleven points over doing nothing, five over the best if-statement. On a fixed review
budget that's worth having, but it isn't the strongest thing on this table and I'm not going to
pretend it is.

The spread is the actual argument. Look at the worst-fold column. "Oldest page" gets 0.80 on one
client fold and 0.24 on another. "Stalest page" runs 0.46 to 0.86. The model sits between 0.66 and
0.80 on those same folds. So the choice isn't "a rule that works versus a model that works a bit
better." It's "a rule that happens to fit whichever client you tested it on" versus a scorer that
holds up on clients it has never seen. Across 32 clients, with a queue going out every Monday,
that stability is the point.

The rules don't even agree with each other. Their top-50 queues overlap by 0.6 pages out of 50.
Five reasonable-sounding if-statements, five almost completely different weeks of work for the
same editor. There's no obvious rule sitting here waiting to be found, and that's what "too messy
for an if-statement" actually looks like in this data.

What this isn't is a result. The label is still the snapshot proxy, each fold's Precision@50 rests
on 50 rows, and two of my features are measured at the wrong end of the window. The number I'll
defend at the end of this lane will come from the warehouse with a forward window I built myself.
All this cell establishes is that the pattern is multi-signal and client-dependent, which is what
the section asked.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**The frame in one paragraph.** For a content strategist who can review about 50 pages a week,
deciding which pages to open first, I'll build a ranked review queue with reason codes from the
FlyRank content dataset, scoring each page's risk of losing search impressions over the next 30
days, measured by Precision@50 averaged across client-grouped folds. A wrong call costs an
editor's hours on a healthy page, plus another week of decline on the page they should have opened
instead. A plain rule isn't enough because the single-signal rules swing between 0.24 and 0.86
depending on which client you test them on, and their top-50 queues overlap by less than one page.
I'll only claim observed, directional, decision-support results.